# Project - Airline AI Assistant

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import json

In [12]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY environment variable is not set.")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY environment variable is not set.")


In [13]:
MODEL_groq = "llama-3.3-70b-versatile" # groq
MODEL_gemini ="gemini-3-flash-preview"  # gemini

In [14]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [15]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content": system_message}] + history + [{"role":"user", "content": message}]  
    response = groq.chat.completions.create(model=MODEL_groq, messages=messages)
    return response.choices[0].message.content


gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [17]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL_gemini, messages=messages)
    return response.choices[0].message.content


gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


# Tools

In [18]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

In [19]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [20]:
get_ticket_price("xyz")

Tool called for city xyz


'The price of a ticket to xyz is Unknown ticket price'

In [21]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [22]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [23]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

# Getting LLM to use our tool

## 🎯 Single-Execution Tool Pattern
This implementation is designed for **high-precision, single-task** scenarios. It simplifies the interaction by focusing strictly on the primary tool identified by the model.

### **How it works:**
* **Fixed Indexing:** By using `message.tool_calls[0]`, the system explicitly handles only the first suggested tool, ignoring any secondary or parallel requests.
* **Streamlined Logic:** Unlike the list-based approach, this returns a single dictionary instead of a list, making the data flow very easy to trace and debug.
* **Predictable State:** The `if` statement ensures a maximum of one "Assistant -> Tool -> Assistant" cycle, keeping response times fast and preventing unintended multi-step execution.

> **Best for:** Applications with a single primary function, such as a dedicated price bot or a simple status checker where multiple simultaneous requests are unlikely.

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL_gemini, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = gemini.chat.completions.create(model=MODEL_gemini, messages=messages)

        for message in messages:
            print(message)


    return response.choices[0].message.content



In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'hi there'}
{'role': 'assistant', 'content': 'Hello! How can I assist you with your FlightAI travel plans today?'}
{'role': 'user', 'content': "i'd like to go to trip "}
{'role': 'assistant', 'content': 'That sounds exciting! Where would you like to travel to?'}
{'role': 'user', 'content': 'i/d like to go to london '}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='igr6rknb', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function', extra_content={'google': {'thought_signature': 'EuoCCucCAb4+9vslfe/JURIBbSyhiIIEvuiBznHmxswPa/fZwCUCXHOw7CRYGb3GEgyqSm63C8SB

# Making Improvements

## ⚡ Linear Single-Turn Tool Interaction
This implementation uses a standard **`if` statement** to handle tool execution. It is a clean, straightforward approach for deterministic tasks.

### **How it works:**
* **One-Shot Execution:** If the model identifies a need for a tool (like `get_ticket_price`), it triggers the call exactly once.
* **Final Synthesis:** After the tool returns the data, the model performs one final pass to generate a user-friendly response.
* **Efficiency:** It prevents infinite loops and minimizes API costs by limiting the conversation to a maximum of two model turns per user prompt.

> **Best for:** Simple utility bots, calculators, or scenarios with clearly defined, independent tool requirements.

In [ ]:

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL_groq, messages=messages, tools=tools)


# doesnt allow more than 1 set of tool call sequentially
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL_groq, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content


gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'find the price for flights to london and only if its under$1000 please check price to paris'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='248gs4s1e', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to London is $799', 'tool_call_id': '248gs4s1e'}


## 🔄 Recursive Multi-Turn Tool Interaction
This implementation uses a **`while` loop** to handle complex, multi-step reasoning. It is designed for tasks where one tool call might not be enough to satisfy the user's request.

### **How it works:**
* **Continuous Reasoning:** The loop checks the `finish_reason`. If the model requests `tool_calls`, the system executes them and feeds the results back.
* **Chained Logic:** The model can use the output of the first tool to inform the second tool call (e.g., fetching a city's ID first, then fetching the ticket price for that ID).
* **Dynamic Termination:** The loop only breaks once the LLM decides it has enough information to provide a final text response.

> **Best for:** Complex AI Agents and workflows where steps are interdependent.

In [ ]:

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL_groq, messages=messages, tools=tools)



    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL_groq, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content


gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Tool called for city London
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'find the price for flights to london and only if its under$1000 please check price to paris'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='4wts9k1zq', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to London is $799', 'tool_call_id': '4wts9k1zq'}
Tool called for city London
Tool called for city Paris
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know th

### 🤖 Building an Autonomous Security Auditor

"Security Sentinel":
1. Receives a network target.
2. Uses a `while` loop to iteratively poll system ports.
3. Analyzes the 'Tool' output to identify unencrypted services (like FTP or HTTP).
4. Provides a human-readable risk assessment.

This moves AI from a simple "Chatbot" to an "Actionable Agent" that can interact with real-world infrastructure.

In [ ]:
def check_port_status(ip, port):
    # In a real lab, 'socket' library will be used to check the port status.
    # For this example, we simulate common security findings
    vulnerable_ports = {
        "21": "Open (Insecure: FTP is unencrypted)",
        "22": "Open (Secure: SSH)",
        "80": "Open (Insecure: HTTP)",
        "443": "Open (Secure: HTTPS)"
    }
    status = vulnerable_ports.get(str(port), "Closed")
    return f"Port {port} on {ip} is {status}"



def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "check_port_status":
            arguments = json.loads(tool_call.function.arguments)
            ip = arguments.get('ip')
            port = arguments.get('port')
            status = check_port_status(ip, port)
            responses.append({
                "role": "tool",
                "content": status,
                "tool_call_id": tool_call.id
            })
    return responses

system_message = """
PROTOCOL: SECURITY_AUDITOR_V5
GOAL: Identify network vulnerabilities via iterative port analysis.
CONSTRAINTS: 
1. Do not speculate. Use the 'check_port_status' tool for evidence.
2. If a port is unencrypted (e.g., 21, 80), flag it as 'CRITICAL RISK'.
3. If a port is encrypted (e.g., 22, 443), flag it as 'COMPLIANT'.
4. Provide a final summary table of findings.
"""

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL_groq, messages=messages, tools=tools)



    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL_groq, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content


gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()
